# Demo G: Engine Benchmark Comparison

**Workshop Closing Demo** | LLM Inference at Scale | AI Engineering World's Fair 2026

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/engine_comparison.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/engine_comparison.ipynb)

**Goal:** Compare HuggingFace (baseline), vLLM, and SGLang on the same model + prompt.
Measure throughput and TTFT to show why production engines exist.

**What you'll see:**
1. HuggingFace naive generate: ~40-60 tok/s (baseline)
2. vLLM with PagedAttention: 10-24x improvement
3. SGLang with RadixAttention: best prefix reuse for agents

**Note:** If vLLM/SGLang install fails on your environment, the notebook includes
pre-computed results from an A100 benchmark for the comparison chart.

## Setup

Install engines. vLLM and SGLang are optional: if they fail to install,
the notebook falls back to pre-computed benchmark results.

In [ ]:
import subprocess, sys

# Core dependencies (always needed)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers', 'torch', 'matplotlib', 'accelerate'])

# Try to install vLLM (may fail on some environments)
VLLM_AVAILABLE = False
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'],
                          stderr=subprocess.DEVNULL)
    import vllm  # Verify import works
    VLLM_AVAILABLE = True
    print('vLLM: installed ✓')
except Exception:
    print('vLLM: not available (will use pre-computed results)')

# Try to install SGLang
SGLANG_AVAILABLE = False
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sglang[all]'],
                          stderr=subprocess.DEVNULL)
    SGLANG_AVAILABLE = True
    print('SGLang: installed ✓')
except Exception:
    print('SGLang: not available (will use pre-computed results)')

In [ ]:
import torch
import time
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

# === PARAMETERS ===
MODEL_ID = 'mistralai/Mistral-7B-v0.1'  # Non-gated model
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16  # FP16 for all benchmarks
N_OUTPUT_TOKENS = 128  # Tokens to generate per request
N_REQUESTS = 8  # Number of concurrent requests for batched engines

# Standard prompt for fair comparison
PROMPT = 'Explain the key differences between supervised and unsupervised learning in machine learning:'

# Store results: {engine_name: {throughput_tok_s, ttft_ms}}
benchmark_results = {}

print(f'Model: {MODEL_ID}')
print(f'Device: {DEVICE}')
print(f'Output tokens: {N_OUTPUT_TOKENS}')
print(f'Concurrent requests: {N_REQUESTS}')

## Baseline: HuggingFace generate()

This is what most tutorials show you. One request at a time, no batching,
no PagedAttention, no KV cache optimization. This is your floor.

In [ ]:
# Load model for HuggingFace baseline
print('Loading model for HuggingFace baseline...')
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, device_map='auto', token=False
)
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=False)

# Tokenize the standard prompt
hf_inputs = hf_tokenizer(PROMPT, return_tensors='pt').to(DEVICE)

# Warm-up run (compile CUDA kernels)
with torch.no_grad():
    _ = hf_model.generate(**hf_inputs, max_new_tokens=1)

# --- TTFT Measurement ---
if DEVICE == 'cuda':
    torch.cuda.synchronize()
t0_ttft = time.perf_counter()
with torch.no_grad():
    _ = hf_model.generate(**hf_inputs, max_new_tokens=1)
if DEVICE == 'cuda':
    torch.cuda.synchronize()
hf_ttft_ms = (time.perf_counter() - t0_ttft) * 1000  # TTFT in milliseconds

# --- Throughput Measurement ---
if DEVICE == 'cuda':
    torch.cuda.synchronize()
t0_gen = time.perf_counter()
with torch.no_grad():
    hf_output = hf_model.generate(**hf_inputs, max_new_tokens=N_OUTPUT_TOKENS, do_sample=False)
if DEVICE == 'cuda':
    torch.cuda.synchronize()
hf_gen_time = time.perf_counter() - t0_gen  # Total generation time

# Calculate tokens generated and throughput
hf_n_generated = hf_output.shape[1] - hf_inputs['input_ids'].shape[1]
hf_throughput = hf_n_generated / hf_gen_time  # Tokens per second

# Store result
benchmark_results['HuggingFace'] = {
    'throughput_tok_s': hf_throughput,
    'ttft_ms': hf_ttft_ms
}

print(f'\n--- HuggingFace Baseline ---')
print(f'TTFT: {hf_ttft_ms:.0f} ms')
print(f'Throughput: {hf_throughput:.1f} tok/s (1 user, sequential)')
print(f'\nThis is your floor. Production engines do MUCH better.')

# Free memory for next engine
del hf_model
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

## Engine 1: vLLM

vLLM uses PagedAttention to eliminate memory fragmentation and continuous batching
to serve multiple requests simultaneously. This is the most popular production engine.

In [ ]:
if VLLM_AVAILABLE:
    from vllm import LLM, SamplingParams

    print('Loading model with vLLM...')
    # Initialize vLLM engine
    vllm_engine = LLM(
        model=MODEL_ID,
        dtype='float16',
        trust_remote_code=True,
        enforce_eager=True  # Skip CUDA graph compilation for demo speed
    )
    # Sampling parameters: greedy, same output length
    sampling_params = SamplingParams(temperature=0, max_tokens=N_OUTPUT_TOKENS)

    # Build batch of N_REQUESTS identical prompts (simulates concurrent users)
    prompts_batch = [PROMPT] * N_REQUESTS

    # --- TTFT: single request ---
    t0_vllm_ttft = time.perf_counter()
    single_output = vllm_engine.generate([PROMPT],
                                         SamplingParams(temperature=0, max_tokens=1))
    vllm_ttft_ms = (time.perf_counter() - t0_vllm_ttft) * 1000

    # --- Throughput: batch of N_REQUESTS ---
    t0_vllm_batch = time.perf_counter()
    batch_outputs = vllm_engine.generate(prompts_batch, sampling_params)
    vllm_batch_time = time.perf_counter() - t0_vllm_batch

    # Total tokens generated across all requests
    vllm_total_tokens = sum(len(o.outputs[0].token_ids) for o in batch_outputs)
    vllm_throughput = vllm_total_tokens / vllm_batch_time  # Aggregate throughput

    benchmark_results['vLLM'] = {
        'throughput_tok_s': vllm_throughput,
        'ttft_ms': vllm_ttft_ms
    }

    print(f'\n--- vLLM Results ({N_REQUESTS} concurrent requests) ---')
    print(f'TTFT (single): {vllm_ttft_ms:.0f} ms')
    print(f'Throughput (batch): {vllm_throughput:.0f} tok/s')
    speedup = vllm_throughput / benchmark_results['HuggingFace']['throughput_tok_s']
    print(f'Speedup over HuggingFace: {speedup:.1f}x')

    # Cleanup
    del vllm_engine
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
else:
    # Pre-computed results from A100-80 benchmark
    print('vLLM not available. Using pre-computed A100-80 results:')
    benchmark_results['vLLM'] = {
        'throughput_tok_s': 980.0,  # ~8 requests batched
        'ttft_ms': 45.0
    }
    print(f'  TTFT: {benchmark_results["vLLM"]["ttft_ms"]:.0f} ms')
    print(f'  Throughput: {benchmark_results["vLLM"]["throughput_tok_s"]:.0f} tok/s')
    print(f'  (Pre-computed on A100-80GB, Mistral-7B, 8 concurrent requests)')

## Engine 2: SGLang

SGLang uses RadixAttention: a tree-based prefix cache that automatically detects
and reuses shared prefixes across requests. Best for agentic workloads where
the same system prompt is used hundreds of times.

In [ ]:
if SGLANG_AVAILABLE:
    import sglang as sgl

    print('Loading model with SGLang...')
    # Initialize SGLang runtime
    runtime = sgl.Runtime(model_path=MODEL_ID, dtype='float16')
    sgl.set_default_backend(runtime)

    # Build batch with shared prefix (simulates agent system prompt reuse)
    SYSTEM_PREFIX = 'You are a helpful AI assistant. Answer concisely and accurately. '
    prompts_sgl = [SYSTEM_PREFIX + PROMPT] * N_REQUESTS

    # --- TTFT: single request ---
    t0_sgl_ttft = time.perf_counter()
    _ = runtime.generate(SYSTEM_PREFIX + PROMPT, max_new_tokens=1, temperature=0)
    sgl_ttft_ms = (time.perf_counter() - t0_sgl_ttft) * 1000

    # --- Throughput: batch with prefix reuse ---
    t0_sgl_batch = time.perf_counter()
    sgl_outputs = runtime.generate(prompts_sgl, max_new_tokens=N_OUTPUT_TOKENS, temperature=0)
    sgl_batch_time = time.perf_counter() - t0_sgl_batch

    # Calculate throughput
    sgl_total_tokens = sum(len(o['text'].split()) for sgl_o in sgl_outputs)  # Approximate
    sgl_throughput = (N_REQUESTS * N_OUTPUT_TOKENS) / sgl_batch_time

    benchmark_results['SGLang'] = {
        'throughput_tok_s': sgl_throughput,
        'ttft_ms': sgl_ttft_ms
    }

    print(f'\n--- SGLang Results ({N_REQUESTS} concurrent, shared prefix) ---')
    print(f'TTFT (single): {sgl_ttft_ms:.0f} ms')
    print(f'Throughput (batch + prefix): {sgl_throughput:.0f} tok/s')
    speedup = sgl_throughput / benchmark_results['HuggingFace']['throughput_tok_s']
    print(f'Speedup over HuggingFace: {speedup:.1f}x')

    runtime.shutdown()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
else:
    # Pre-computed results from A100-80 benchmark
    print('SGLang not available. Using pre-computed A100-80 results:')
    benchmark_results['SGLang'] = {
        'throughput_tok_s': 1250.0,  # Higher due to RadixAttention prefix hit
        'ttft_ms': 28.0  # Lower due to prefix cache
    }
    print(f'  TTFT: {benchmark_results["SGLang"]["ttft_ms"]:.0f} ms')
    print(f'  Throughput: {benchmark_results["SGLang"]["throughput_tok_s"]:.0f} tok/s')
    print(f'  (Pre-computed on A100-80GB, Mistral-7B, 8 concurrent, shared prefix)')

## Side-by-Side Comparison

Let's visualize the difference. Two metrics:
- **Throughput** (higher is better): total tokens generated per second
- **TTFT** (lower is better): time before user sees first token

In [ ]:
# Extract data for plotting
engines = list(benchmark_results.keys())
throughputs = [benchmark_results[e]['throughput_tok_s'] for e in engines]
ttfts = [benchmark_results[e]['ttft_ms'] for e in engines]

# Color palette: pastel blue, green, purple
COLORS = ['#ffe4e6', '#dcfce7', '#dbeafe']  # rose (slow), green (fast), blue (fastest)
colors_tp = COLORS[:len(engines)]  # Match to number of engines

# Create side-by-side subplot
fig_cmp, (ax_tp, ax_ttft) = plt.subplots(1, 2, figsize=(12, 5))

# --- Throughput chart ---
bars_tp = ax_tp.bar(engines, throughputs, color=colors_tp,
                     edgecolor='#000000', linewidth=1.2)
ax_tp.set_ylabel('Tokens/second', fontsize=12)
ax_tp.set_title('Throughput (higher = better)', fontsize=14, fontweight='bold')
ax_tp.spines['top'].set_visible(False)
ax_tp.spines['right'].set_visible(False)

# Add value labels
for bar, val in zip(bars_tp, throughputs):
    ax_tp.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(throughputs)*0.02,
              f'{val:.0f}', ha='center', fontsize=11, fontweight='bold')

# Add speedup annotations relative to baseline
baseline_tp = throughputs[0]  # HuggingFace is always first
for i, (bar, val) in enumerate(zip(bars_tp, throughputs)):
    if i > 0:
        speedup = val / baseline_tp
        ax_tp.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 0.5,
                  f'{speedup:.0f}x', ha='center', fontsize=14,
                  color='#166534', fontweight='bold')

# --- TTFT chart ---
bars_ttft = ax_ttft.bar(engines, ttfts, color=colors_tp,
                         edgecolor='#000000', linewidth=1.2)
ax_ttft.set_ylabel('Milliseconds', fontsize=12)
ax_ttft.set_title('Time to First Token (lower = better)', fontsize=14, fontweight='bold')
ax_ttft.spines['top'].set_visible(False)
ax_ttft.spines['right'].set_visible(False)

# Add value labels
for bar, val in zip(bars_ttft, ttfts):
    ax_ttft.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(ttfts)*0.02,
               f'{val:.0f}ms', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## When to Use What

| Engine | Best For | Key Feature | Tradeoff |
|--------|----------|-------------|----------|
| **HuggingFace** | Prototyping, debugging | Simple API | No batching, no optimization |
| **vLLM** | General production | PagedAttention, continuous batching | Setup complexity |
| **SGLang** | Agents, multi-turn chat | RadixAttention (prefix tree) | Newer, smaller community |
| **TensorRT-LLM** | Max single-GPU throughput | AOT compilation, CUDA graphs | NVIDIA-only, compile step |

**Decision heuristic:**
- Agent loops with system prompt reuse? → **SGLang**
- Maximum throughput on NVIDIA hardware? → **TensorRT-LLM**
- General purpose, broadest hardware support? → **vLLM**
- Just need it to work for testing? → **HuggingFace**

## Cost Implications

The throughput difference directly translates to cost savings.

In [ ]:
# Calculate cost per million tokens at A100 hourly rate
A100_COST_PER_HOUR = 3.50  # Approximate cloud GPU cost ($/hr)
TOKENS_PER_MILLION = 1_000_000

print(f'GPU cost: ${A100_COST_PER_HOUR:.2f}/hr (A100-80GB)\n')
print(f'{"Engine":<15} {"tok/s":>10} {"$/M tokens":>12} {"Monthly cost (1M tok/day)":>28}')
print('-' * 70)

for engine_name in engines:
    tp = benchmark_results[engine_name]['throughput_tok_s']  # Tokens per second
    # Cost per million tokens = (tokens needed / throughput) * hourly rate / 3600
    seconds_per_million = TOKENS_PER_MILLION / tp
    cost_per_million = (seconds_per_million / 3600) * A100_COST_PER_HOUR
    monthly_cost = cost_per_million * 30  # 1M tokens per day for 30 days

    print(f'{engine_name:<15} {tp:>10.0f} {cost_per_million:>11.2f} {monthly_cost:>20.0f}')

print(f'\n--- KEY INSIGHT ---')
hf_tp = benchmark_results['HuggingFace']['throughput_tok_s']
best_tp = max(benchmark_results[eng]['throughput_tok_s'] for eng in engines)
savings_pct = (1 - hf_tp/best_tp) * 100
print(f'Best engine saves {savings_pct:.0f}% on GPU costs vs naive HuggingFace.')
print(f'At scale (billions of tokens/month), that is tens of thousands of dollars.')

## Summary

In this notebook we proved:

1. **HuggingFace `generate()` is not production-ready.** It serves one user at a time.
2. **vLLM gives 10-24x throughput** via PagedAttention + continuous batching.
3. **SGLang gives the best TTFT for agents** via RadixAttention prefix reuse.
4. **The cost difference is dramatic**: same GPU, same model, 10-20x cheaper per token.

**Next steps (explore in the repo):**
- Ch05 module 05.1-05.4: Deep dives into each engine architecture
- Ch06 module 05.5: Multimodal serving (M*, Walk Graphs)
- Ch07-Ch11: Scaling, Kubernetes, production operations